# Gold 06 — Saved Model Validation / Test-Only Review

This notebook is meant to validate the saved Gold model artifacts without retraining. It should be run after the Gold training notebooks have already produced model files, thresholds, and the Gold test split. The purpose is to prove that the model can be loaded from disk and scored against the held-out test data as a separate testing step.


## 1. Imports and shared notebook context

This cell imports the validation dependencies and loads the shared notebook context. The validation stage uses the same project path, logging, config, and ledger pattern as the rest of the notebooks, but it uses a separate recipe ID so the output is clearly test-only.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from utils.core.notebook_context import load_notebook_context


In [ ]:

CTX = load_notebook_context(
    stage="gold_model_validation",
    recipe_id="gold_saved_model_validation__v001",
    dataset="pump",
    mode="test",
    profile="default",
    logger_child_name="capstone.gold_model_validation",
    log_filename="gold_model_validation.log",
)

paths = CTX.paths
CONFIG = CTX.config
STAGE_CFG = CTX.stage_config
FILENAMES = CTX.filenames
VERSIONS_CFG = CTX.versions
PIPELINE = CTX.pipeline
logger = CTX.logger
ledger = CTX.ledger

logger.info("Gold saved-model validation notebook initialized")
ledger.add(kind="step", step="validation_context_loaded", message="Loaded test-mode context for saved model validation", logger=logger)


## 2. Resolve validation input and output paths

This cell defines the expected locations for the saved model files, threshold files, Gold test split, and validation outputs. The notebook should fail loudly if a required model or test dataset is missing because this stage is supposed to validate previously trained artifacts rather than rebuild them.


In [ ]:
PROJECT_ROOT = paths.root
DATA_ROOT = paths.data
ARTIFACTS_ROOT = paths.artifacts
MODELS_ROOT = paths.models

GOLD_DATA_DIR = DATA_ROOT / "gold"
MODEL_DIR = MODELS_ROOT / "pump"
VALIDATION_DIR = ARTIFACTS_ROOT / "gold" / "pump" / "model_validation"
VALIDATION_RESULTS_DIR = VALIDATION_DIR / "results"
VALIDATION_SUMMARY_DIR = VALIDATION_DIR / "summaries"
VALIDATION_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

TEST_DATA_CANDIDATES = [
    GOLD_DATA_DIR / "pump__gold__test.parquet",
    GOLD_DATA_DIR / "pump__gold_test.parquet",
    GOLD_DATA_DIR / "pump__gold__model_test.parquet",
]

MODEL_FILES = {
    "baseline": MODEL_DIR / "pump__gold__baseline_isolation_forest.joblib",
    "cascade_default_stage1": MODEL_DIR / "pump__gold__cascade_defaults_stage1_isolation_forest.joblib",
    "cascade_default_stage2": MODEL_DIR / "pump__gold__cascade_defaults_stage2_isolation_forest.joblib",
    "cascade_tuned_stage1": MODEL_DIR / "pump__gold__cascade_tuned_stage1_isolation_forest.joblib",
    "cascade_tuned_stage2": MODEL_DIR / "pump__gold__cascade_tuned_stage2_isolation_forest.joblib",
    "stage3_improved_stage1": MODEL_DIR / "pump__gold__cascade_stage3_improved_stage1_isolation_forest.joblib",
    "stage3_improved_stage2": MODEL_DIR / "pump__gold__cascade_stage3_improved_stage2_isolation_forest.joblib",
}

THRESHOLD_FILES = {
    "baseline": ARTIFACTS_ROOT / "gold" / "pump" / "baseline" / "thresholds" / "pump__gold__baseline_thresholds.json",
    "cascade_default": ARTIFACTS_ROOT / "gold" / "pump" / "cascade_defaults" / "thresholds" / "pump__gold__cascade_defaults_thresholds.json",
    "cascade_tuned": ARTIFACTS_ROOT / "gold" / "pump" / "cascade_tuned" / "thresholds" / "pump__gold__cascade_tuned_thresholds.json",
    "stage3_improved": ARTIFACTS_ROOT / "gold" / "pump" / "cascade_stage3_improved" / "thresholds" / "pump__gold__cascade_stage3_improved_thresholds.json",
}

logger.info("Resolved model validation paths")
ledger.add(kind="step", step="validation_paths_resolved", message="Resolved saved model, threshold, and validation output paths", logger=logger)


## 3. Load the held-out test split

This cell loads the Gold test split. This is intentionally different from the training notebooks: the validation notebook should use the saved test data and should not rerun fitting, resplitting, or preprocessing logic.


In [ ]:
test_path = next((p for p in TEST_DATA_CANDIDATES if p.exists()), None)
if test_path is None:
    raise FileNotFoundError(
        "Could not find the Gold test split. Checked: " + ", ".join(str(p) for p in TEST_DATA_CANDIDATES)
    )

test_dataframe = pd.read_parquet(test_path)

logger.info("Loaded Gold test split", extra={"path": str(test_path), "rows": len(test_dataframe), "columns": len(test_dataframe.columns)})
ledger.add(kind="step", step="test_split_loaded", message=f"Loaded Gold test split from {test_path} with {len(test_dataframe):,} rows", logger=logger)

display(test_dataframe.head())


## 4. Identify label and feature columns

This cell finds the test label and feature columns. It excludes obvious metadata, target, status, timestamp, split, and prediction columns so model scoring uses the same style of numeric feature matrix expected by the saved Isolation Forest estimators.


In [ ]:
LABEL_CANDIDATES = ["target_flag", "is_anomaly", "is_broken", "broken_flag", "target", "label"]
label_column = next((column for column in LABEL_CANDIDATES if column in test_dataframe.columns), None)
if label_column is None:
    raise KeyError(f"Could not find a binary target label column. Checked: {LABEL_CANDIDATES}")

EXCLUDE_PATTERNS = (
    "target", "label", "status", "timestamp", "datetime", "date", "split", "run_id", "dataset_id",
    "prediction", "pred", "alert", "score", "anomaly", "broken", "recovering",
)
feature_columns = [
    column for column in test_dataframe.select_dtypes(include=[np.number]).columns
    if column != label_column and not any(p in column.lower() for p in EXCLUDE_PATTERNS)
]
if not feature_columns:
    raise ValueError("No numeric model feature columns were found after exclusions.")

y_test = test_dataframe[label_column].astype(int).to_numpy()
X_test = test_dataframe[feature_columns].replace([np.inf, -np.inf], np.nan).fillna(0.0)

logger.info("Prepared validation feature matrix", extra={"label_column": label_column, "feature_count": len(feature_columns), "rows": len(X_test)})
ledger.add(kind="step", step="validation_features_prepared", message=f"Prepared {len(feature_columns)} validation feature columns using label {label_column}", logger=logger)

pd.DataFrame({"feature_column": feature_columns}).head(20)


## 5. Load saved models and thresholds

This cell loads the saved model files and threshold JSON files from disk. It does not call `.fit()`. If this cell fails, the project either has missing model artifacts or the model directory does not match the notebook outputs.


In [ ]:
models: Dict[str, Any] = {}
missing_models = []
for name, path in MODEL_FILES.items():
    if path.exists():
        models[name] = joblib.load(path)
    else:
        missing_models.append(str(path))

thresholds: Dict[str, Dict[str, Any]] = {}
for name, path in THRESHOLD_FILES.items():
    if path.exists():
        with path.open("r", encoding="utf-8") as f:
            thresholds[name] = json.load(f)

if missing_models:
    raise FileNotFoundError("Missing saved model files: " + "; ".join(missing_models))

logger.info("Loaded saved models and threshold files", extra={"models": sorted(models), "threshold_sets": sorted(thresholds)})
ledger.add(kind="step", step="saved_models_loaded", message=f"Loaded {len(models)} saved model files and {len(thresholds)} threshold files", logger=logger)

sorted(models), sorted(thresholds)


## 6. Define reusable scoring helpers

This cell defines the helper functions used to convert Isolation Forest model outputs into binary alert flags and metrics. The helpers are intentionally small so the validation logic is clear and easy to compare with Gold 04.


In [ ]:
def anomaly_score(model: Any, X: pd.DataFrame) -> np.ndarray:
    """Return a positive anomaly-strength score for an Isolation Forest model.

    scikit-learn's Isolation Forest returns lower values for more abnormal rows. This helper flips the sign so larger values mean stronger anomaly evidence, which makes threshold comparisons easier to read.
    """
    if hasattr(model, "decision_function"):
        return -model.decision_function(X)
    if hasattr(model, "score_samples"):
        return -model.score_samples(X)
    raise TypeError(f"Model {type(model).__name__} does not expose decision_function or score_samples")


def threshold_from_summary(threshold_payload: Dict[str, Any], fallback_quantile: float, scores: np.ndarray) -> float:
    """Resolve a threshold from a threshold JSON payload or from score quantiles.

    The model notebooks save threshold details under slightly different keys. This helper checks common names first, then falls back to a quantile-based threshold so validation can still run while the threshold contract is being standardized.
    """
    possible_keys = ["threshold", "score_threshold", "stage1_threshold", "stage2_threshold", "final_threshold"]
    for key in possible_keys:
        value = threshold_payload.get(key)
        if isinstance(value, (int, float)):
            return float(value)
    return float(np.quantile(scores, fallback_quantile))


def calculate_binary_metrics(model_name: str, y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray) -> Dict[str, Any]:
    """Calculate binary classification metrics for one saved-model validation run."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    row = {
        "model": model_name,
        "test_rows": int(len(y_true)),
        "alerts": int(y_pred.sum()),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }
    if len(np.unique(y_true)) > 1:
        row["roc_auc"] = float(roc_auc_score(y_true, y_score))
        row["average_precision"] = float(average_precision_score(y_true, y_score))
    else:
        row["roc_auc"] = None
        row["average_precision"] = None
    return row

logger.info("Defined saved-model scoring helpers")
ledger.add(kind="step", step="validation_helpers_defined", message="Defined scoring and metric helper functions", logger=logger)


## 7. Score baseline and cascade model files

This cell applies the saved models to the held-out test split. For the first pass, the baseline and each saved stage model are validated individually. If the cascade notebooks save exact stage-gate outputs or threshold manifests, this cell can be extended to reconstruct full cascade gating exactly.


In [ ]:
validation_rows: List[Dict[str, Any]] = []
score_frames: List[pd.DataFrame] = []

for model_name, model in models.items():
    scores = anomaly_score(model, X_test)
    threshold_group = model_name
    if "cascade_default" in model_name:
        threshold_group = "cascade_default"
    elif "cascade_tuned" in model_name:
        threshold_group = "cascade_tuned"
    elif "stage3_improved" in model_name:
        threshold_group = "stage3_improved"
    threshold = threshold_from_summary(thresholds.get(threshold_group, {}), 0.95, scores)
    y_pred = (scores >= threshold).astype(int)
    metrics = calculate_binary_metrics(model_name, y_test, y_pred, scores)
    metrics["threshold"] = threshold
    validation_rows.append(metrics)
    score_frames.append(pd.DataFrame({
        "row_index": np.arange(len(test_dataframe)),
        "model": model_name,
        "score": scores,
        "prediction": y_pred,
        "label": y_test,
    }))

validation_dataframe = pd.DataFrame(validation_rows).sort_values(["f1", "precision"], ascending=False)
validation_scores_dataframe = pd.concat(score_frames, ignore_index=True)

logger.info("Scored saved models against test split", extra={"model_count": len(models), "rows": len(validation_scores_dataframe)})
ledger.add(kind="step", step="saved_models_scored", message="Scored saved models against the held-out Gold test split", logger=logger)

validation_dataframe


## 8. Save validation outputs

This cell writes the validation table and row-level score table. These files can be compared to Gold 04 to confirm that saved-model testing is aligned with the notebook training outputs.


In [ ]:
validation_table_path = VALIDATION_RESULTS_DIR / "pump__gold__saved_model_validation_metrics.csv"
validation_scores_path = VALIDATION_RESULTS_DIR / "pump__gold__saved_model_validation_scores.csv"
validation_summary_path = VALIDATION_SUMMARY_DIR / "pump__gold__saved_model_validation_summary.json"

validation_dataframe.to_csv(validation_table_path, index=False)
validation_scores_dataframe.to_csv(validation_scores_path, index=False)

summary_payload = {
    "stage": "gold_model_validation",
    "mode": "test",
    "test_data_path": str(test_path),
    "test_rows": int(len(test_dataframe)),
    "label_column": label_column,
    "feature_count": int(len(feature_columns)),
    "models_validated": sorted(models),
    "metric_output": str(validation_table_path),
    "score_output": str(validation_scores_path),
}
validation_summary_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")

logger.info("Saved model validation outputs", extra={"metrics": str(validation_table_path), "scores": str(validation_scores_path)})
ledger.add(kind="result", step="saved_model_validation_outputs", message="Saved saved-model validation metrics and scores", logger=logger)

validation_table_path, validation_scores_path, validation_summary_path


## 9. Validation interpretation

This final cell displays the saved-model validation metrics and identifies the best F1 and lowest-alert options. The final Task 3 report should still use Gold 04 as the main comparison table unless this saved-model validation table is confirmed to match the Gold 04 metric contract exactly.


In [ ]:
best_f1 = validation_dataframe.iloc[0].to_dict() if not validation_dataframe.empty else {}
lowest_alert = validation_dataframe.sort_values("alerts").iloc[0].to_dict() if not validation_dataframe.empty else {}

interpretation = {
    "best_f1_model": best_f1.get("model"),
    "best_f1": best_f1.get("f1"),
    "lowest_alert_model": lowest_alert.get("model"),
    "lowest_alert_count": lowest_alert.get("alerts"),
    "note": "Use this notebook as a saved-artifact validation check. Keep Gold 04 as the final model comparison until this validation reproduces the same full cascade gating contract.",
}

logger.info("Completed saved-model validation interpretation", extra=interpretation)
ledger.add(kind="result", step="saved_model_validation_complete", message="Completed saved-model validation notebook", logger=logger)

interpretation
